# 05 - Follow-up FU1: can the rejected clean blocks of the 12-label runs be learned?
Pre-registration: `experiments/prereg_followup_fu1.md` (committed before any FU1 run). Code: `s9_followup.py`.

The all12 runs rejected **clean** updates. At E1, DNS validation rows were predicted LDAP. At E5, SSDP rows were
predicted UDP. s3 then skips such a block for good. This notebook, deciding on **validation only**:
0. measures 1-nearest-neighbour separability of DNS/LDAP and SSDP/UDP (a ceiling hint, decides nothing);
1. probes E1 of the clean arm: modest training / class-balance changes, one factor at a time, 3 seeds;
2. probes E5: balanced class weights, or distillation from the last accepted model;
3. applies the pre-registered selection rule (`decide`);
4. runs the clean schedule with the retry policy (C1 -> C2 -> deferred D1). No test is scored;
5. writes `fu1_val.zip` (validation JSON/CSV) and `fu1_restore.zip` (validated models + sha256 manifest), then
   **STOP**: `fu1_val.zip` is committed to git. Only then is test scored, once (cell 8, needs the commit sha).
   If Kaggle closes in between, a new session restores `fu1_restore.zip` (cell 7) and scores test without retraining.

Read only: the all12 runs and the frozen teacher. Everything is written under `/kaggle/working/nids_out/followup/`.

**Inputs (Add Input):** the output of notebook 04's second run, which holds `nids_out/` with `clean_*.parquet`,
`all12/` and `runs/{cii,ci}/all12_*_s42`. Add it via Your Work -> notebook 04, or upload `results_04_2.zip` as a
dataset. Also turn **Internet ON** (git clone of the code). Accelerator: **GPU T4**.

Run interactively, not "Save & Run All". Cells 7-8 do nothing unless `RESTORE` / `DECISIONS_COMMIT` are set.

In [ ]:
# 1. Locate code and the notebook-04 outputs (read only)
import glob, json, os, subprocess, sys, time, zipfile
from pathlib import Path
import pandas as pd
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)

CLONE = Path("/kaggle/working/code")
if not CLONE.exists() and os.environ.get("NIDS_CLONE", "1") == "1":
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/DevChi1l/cicddos2019-continual-nids",
                    str(CLONE)], check=False)
# marker s9_followup.py: an older code copy (e.g. inside the notebook-04 output) is never picked
CODE_DIR = os.environ.get("NIDS_CODE") or next(
    (str(Path(p).parent) for p in [str(CLONE / "s9_followup.py")] +
     sorted(glob.glob("/kaggle/input/**/s9_followup.py", recursive=True)) if Path(p).exists()), None)
assert CODE_DIR, "code with s9_followup.py not found: turn Internet ON"
sys.path.insert(0, CODE_DIR)
import s5_explain as X
git = subprocess.run(["git", "-C", CODE_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("code:", CODE_DIR, "| git commit:", git)

SRC = X.find_dir("runs/cii/all12_cii_s42/run_manifest.json", "NIDS_FU_SRC")
assert SRC and (SRC / "clean_train.parquet").exists() and (SRC / "runs/ci/all12_ci_s42/run_manifest.json").exists(), \
    "attach the notebook-04 output that contains nids_out/ with clean_*.parquet, all12/ and both all12 runs"
WORK = Path(os.environ.get("NIDS_WORK", "/kaggle/working/nids_out"))
FU = WORK / "followup"
print("source (read only):", SRC, "\nwrites:", FU)

def run(*args):
    cmd = [sys.executable, str(Path(CODE_DIR) / "s9_followup.py"), *map(str, args)]
    print("$", " ".join(cmd[1:]), flush=True)
    subprocess.run(cmd, check=True, cwd=CODE_DIR)

COMMON = ["--src", SRC, "--work", WORK]
SCENARIOS = ["cii", "ci"]

def zip_followup(name, test=False):
    """JSON/CSV under followup/ -> /kaggle/working/<name>. Defined here (not in cell 6) so a restore session,
    which skips cells 2-6, can still package test (cell 9)."""
    dest = WORK.parent / name
    pick = sorted(p for p in FU.rglob("*") if p.is_file() and p.suffix in (".json", ".csv")
                  and (test or p.name != "final_test.json"))
    with zipfile.ZipFile(dest, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in pick:
            zf.write(p, Path("followup") / p.relative_to(FU))
    print(f"{len(pick)} files -> {dest} ({dest.stat().st_size / 1e6:.2f} MB). Download it from the Output panel.")

In [ ]:
# 2. GPU, then step 0: separability (descriptive; nothing is decided from it). Exact float64 distances ->
#    separability_exact.csv. The float32 separability.csv of the 2026-10-21 run is invalid (see 05_fu1/README).
import torch
assert torch.cuda.is_available(), "Settings -> Accelerator -> GPU T4"
print(torch.cuda.get_device_name(0), torch.__version__)
t0 = time.time()
for sc in SCENARIOS:
    run("separability", *COMMON, "--scenario", sc, "--pairs", "DNS:LDAP,SSDP:UDP")
print(f"step 0: {time.time() - t0:.0f} s")

## Steps 1-2: probes (validation only, one factor at a time, seeds 42/43/44)
Each probe starts from the clean arm's saved start model and buffer for that block. `base` with seed 42 must
reproduce the logged s3 C1 (column `reproduces_logged_C1` = 0.0). Finished probes are skipped if the cell is re-run.

In [ ]:
# 3. Step 1: E1 (DNS, LDAP) and step 2: E5 (UDP, UDPLag)
t0 = time.time()
for sc in SCENARIOS:
    run("probe", *COMMON, "--scenario", sc, "--block", 1, "--recipes", "base,ep10,ep20,lr3e-4,balanced")
print(f"step 1: {time.time() - t0:.0f} s"); t0 = time.time()
for sc in SCENARIOS:
    run("probe", *COMMON, "--scenario", sc, "--block", 5, "--recipes", "base,balanced,distill")
print(f"step 2: {time.time() - t0:.0f} s")
for sc in SCENARIOS:
    for b in (1, 5):
        d = pd.read_csv(FU / sc / f"probe_E{b}.csv")
        print(f"\n{sc} E{b}"); print(d.drop(columns=["gate_reasons", "block"]).round(4).to_string(index=False))

In [ ]:
# 4. Step 3: pre-registered selection rule -> followup/decisions.json
run("decide", "--work", WORK, "--new-class-block", 1, "--old-class-block", 5)

In [ ]:
# 5. Step 4: clean schedule with the retry policy, both scenarios (validation decisions only; no test)
t0 = time.time()
for sc in SCENARIOS:
    run("policy", *COMMON, "--scenario", sc, "--decisions", FU / "decisions.json", "--run", f"fu1_retry_{sc}_s42")
print(f"step 4: {time.time() - t0:.0f} s")
for sc in SCENARIOS:
    s = json.loads((FU / sc / f"fu1_retry_{sc}_s42" / "summary.json").read_text())
    print(sc, s["block_status"], "\n   deployed classes:", s["final_classes"])

## STOP - download BOTH zips from the Output panel (cell 6 writes them)
- `fu1_val.zip`: validation files only (JSON/CSV, no checkpoints, no test). Send it; it is committed to git under
  `experiments/results/05_fu1/`.
- `fu1_restore.zip`: the validated final models + decisions + rounds + a sha256 manifest (no test). Keep it; it
  lets test be scored in a NEW session if this one closes.

Test is scored once, only after the commit. In this session: set `DECISIONS_COMMIT` in cell 8 and run cells 8-9.
If the session closed: see the restore cell (7).

In [ ]:
# 6. Validation outputs: fu1_val.zip (JSON/CSV only) and fu1_restore.zip (models + manifest); neither holds test
assert not list(FU.rglob("final_test.json")), "test was already scored in this folder"
zip_followup("fu1_val.zip")
run("bundle", "--work", WORK, "--bundle", WORK.parent / "fu1_restore.zip")

In [ ]:
# 7. RESTORE (only in a NEW session after the old one closed; skip otherwise).
#    New session: Internet ON; Add Input = notebook 04's output (as before) + a private dataset made from
#    fu1_restore.zip. Run cell 1, then this cell with RESTORE = True, then cells 8-9. Never run cells 2-6 here.
RESTORE = False
if RESTORE:
    hits = sorted(glob.glob("/kaggle/input/**/RESTORE_MANIFEST.json", recursive=True)) or \
           sorted(glob.glob("/kaggle/input/**/fu1_restore.zip", recursive=True))
    assert hits, "attach the dataset made from fu1_restore.zip"
    b = Path(hits[0])
    run("restore", "--work", WORK, "--bundle", b.parent if b.name == "RESTORE_MANIFEST.json" else b)
else:
    print("restore skipped (RESTORE = False)")

In [ ]:
# 8. Test, ONCE. Disabled until DECISIONS_COMMIT is the sha of the commit that holds fu1_val.zip's files.
#    score-test refuses unless decisions.json, summary.json and rounds.json here are byte-identical to that
#    commit's copies under experiments/results/05_fu1/ and final.pt has the hash recorded in summary.json.
DECISIONS_COMMIT = ""
if DECISIONS_COMMIT:
    subprocess.run(["git", "-C", CODE_DIR, "fetch", "--depth", "50", "origin", "main"], check=True)
    for sc in SCENARIOS:
        run("score-test", *COMMON, "--scenario", sc, "--run", f"fu1_retry_{sc}_s42",
            "--decisions-commit", DECISIONS_COMMIT, "--repo", CODE_DIR)
else:
    print("skipped: commit the validation files first, then set DECISIONS_COMMIT")

In [ ]:
# 9. Test files (after cell 8 ran; works in the original or a restore session - needs only cell 1):
#    fu1_test.zip = the JSON/CSV under followup/ + final_test.json per run
missing = [sc for sc in SCENARIOS if not (FU / sc / f"fu1_retry_{sc}_s42" / "final_test.json").exists()]
if len(missing) == len(SCENARIOS):
    print("no test scored yet")
else:
    assert not missing, f"test not scored for {missing}: run cell 8"
    zip_followup("fu1_test.zip", test=True)